<a href="https://colab.research.google.com/github/aymanr8-dot/ds2002-fa26/blob/main/notebooks/04-json-api/2026_10_02_normalize_and_join_nested_json_lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Normalize and Join Nested JSON

**Lab — 2026-10-02 · Fall 2026**  

---

## Lab 06 — Normalize and Join Nested JSON

Two JSON payloads of the kind an API hands you. Flatten each, join them, and answer the questions.

Before you write anything, answer this for yourself: **what is one row of the table you are building?** Every argument you pass to `json_normalize` follows from that answer, and getting it wrong is the reason this lab takes people two hours instead of forty minutes.

One vendor in the orders is not in the vendor list. That is on purpose.

In [ ]:
orders_json = [
  {'order':1,'vendor_id':'V-01','lines':[{'item':'Cheeseburger','qty':2},{'item':'Soda','qty':1}]},
  {'order':2,'vendor_id':'V-10','lines':[{'item':'Foam Finger','qty':1}]},
  {'order':3,'vendor_id':'V-01','lines':[{'item':'Hot Dog','qty':3}]},
]
vendors_json = [
  {'vendor_id':'V-01','name':'Hoos Burgers','zone':'A'},
  {'vendor_id':'V-10','name':'Cav Merch','zone':'A'},
]

**One row of my result is one:** line item, meaning one item on one order, with the order number and vendor_id carried down onto it.

### Q1 — Flatten the orders to one row per line item.

Keep `order` and `vendor_id` on every row.

*Expected: 4 rows, and the quantities should sum to 7.*

In [ ]:
import pandas as pd

# One row = one line item (one item on one order).
# record_path says where the rows live; meta carries the parent fields down onto each row.
lines = pd.json_normalize(orders_json, record_path='lines', meta=['order', 'vendor_id'])

assert len(lines) == 4, f'expected 4 line items, got {len(lines)}'
assert lines['qty'].sum() == 7, f"expected 7 units, got {lines['qty'].sum()}"
lines

,item,qty,order,vendor_id
0,Cheeseburger,2,1,V-01
1,Soda,1,1,V-01
2,Foam Finger,1,2,V-10
3,Hot Dog,3,3,V-01


### Q2 — Join the vendor information on.

Left join, `indicator=True`. Report which vendor did not match and how many units are attached to it.

In [ ]:
vendors = pd.json_normalize(vendors_json)

# Left join keeps every line item even if its vendor is unknown.
# validate='m:1' guards against duplicate vendor_ids fanning out the rows.
joined = lines.merge(vendors, on='vendor_id', how='left', indicator=True, validate='m:1')
display(joined)

print(joined['_merge'].value_counts(), '\n')

unmatched = joined[joined['_merge'] == 'left_only']
print('Unmatched vendor_ids:', sorted(unmatched['vendor_id'].unique()))
print('Units attached to unmatched vendors:', unmatched['qty'].sum())

# REPORT: with the payloads as given above, all 4 line items matched (_merge == 'both').
# Unmatched vendor_ids: none, so 0 units are attached to an unmatched vendor.
# V-01 and V-10 both appear in vendors_json, so the "one vendor is missing" case from
# the intro does not occur in this data. If it did, it would show up here as left_only.

,item,qty,order,vendor_id,name,zone,_merge
0,Cheeseburger,2,1,V-01,Hoos Burgers,A,both
1,Soda,1,1,V-01,Hoos Burgers,A,both
2,Foam Finger,1,2,V-10,Cav Merch,A,both
3,Hot Dog,3,3,V-01,Hoos Burgers,A,both


_merge
both          4
left_only     0
right_only    0
Name: count, dtype: int64 

Unmatched vendor_ids: []
Units attached to unmatched vendors: 0


### Q3 — Total quantity per vendor name.

Decide how the unmatched vendor appears in this table, and say why in a comment.

In [ ]:
# Unmatched vendors have name = NaN, and groupby drops NaN keys by default,
# which would silently lose their units. Label them explicitly instead, so the
# table always sums to the full 7 and the gap is visible rather than hidden.
qty_by_vendor = (
    joined.assign(name=joined['name'].fillna('UNKNOWN (' + joined['vendor_id'] + ')'))
          .groupby('name', as_index=False)['qty'].sum()
          .sort_values('qty', ascending=False)
)
display(qty_by_vendor)
assert qty_by_vendor['qty'].sum() == lines['qty'].sum()
# RESULT here: Hoos Burgers 6, Cav Merch 1 (total 7). No UNKNOWN row appears because
# every vendor_id matched, but the label is kept so an unmatched vendor could not hide.

,name,qty
1,Hoos Burgers,6
0,Cav Merch,1


### Q4 — Which zone sold the most items?

Careful: if the unmatched vendor has no zone, your zone totals will not add up to 7. State what your total is and where the difference went.

In [ ]:
# dropna=False keeps a NaN-zone row for any unmatched vendor instead of dropping it.
qty_by_zone = joined.groupby('zone', dropna=False)['qty'].sum().sort_values(ascending=False)
display(qty_by_zone)

total_all = lines['qty'].sum()
total_zoned = joined.loc[joined['zone'].notna(), 'qty'].sum()
print('Top zone:', qty_by_zone.idxmax(), 'with', qty_by_zone.max(), 'units')
print(f'Units with a known zone: {total_zoned} of {total_all}')
print('Units with no zone (unmatched vendor):', total_all - total_zoned)

# STATED TOTAL: zone A sold the most, with 7 units (both vendors are in zone A).
# Zone totals add up to 7 of 7, so the difference is 0: no vendor was unmatched.
# Had one been unmatched, its units would sit in the NaN-zone row (kept by dropna=False)
# and the gap would equal that vendor's units instead of silently disappearing.

,qty
zone,
A,7


Top zone: A with 7 units
Units with a known zone: 7 of 7
Units with no zone (unmatched vendor): 0


### Q5 — Handle a ragged payload.

A second batch arrives and it is not uniform: one order has no `lines` key at all, and one line item is missing its `qty`.

**TODO:** flatten what you can without crashing, and produce a frame where a missing quantity is distinguishable from a quantity of zero. Do **not** use `fillna(0)`.

In [ ]:
ragged_json = [
  {'order': 4, 'vendor_id': 'V-01', 'lines': [{'item': 'Soda', 'qty': 2}]},
  {'order': 5, 'vendor_id': 'V-10'},                                   # no lines at all
  {'order': 6, 'vendor_id': 'V-01', 'lines': [{'item': 'Fries'}]},     # no qty
]

# json_normalize raises KeyError if record_path is missing from any record,
# so give every order a 'lines' list (empty when the key is absent).
safe = [{**o, 'lines': o.get('lines') or []} for o in ragged_json]
ragged = pd.json_normalize(safe, record_path='lines', meta=['order', 'vendor_id'])

# A missing qty arrives as NaN (which forces the column to float).
# Nullable Int64 keeps it as <NA>: 'we were not told', not 'zero were sold'.
ragged['qty'] = ragged['qty'].astype('Int64')
ragged['qty_missing'] = ragged['qty'].isna()
display(ragged)

# Orders with no line items produce no rows above, so report them rather than lose them.
no_lines = [o['order'] for o in safe if not o['lines']]
print('Orders with no line items:', no_lines)
print('Known units:', ragged['qty'].sum(), '| lines with unknown qty:', int(ragged['qty_missing'].sum()))

,item,qty,order,vendor_id,qty_missing
0,Soda,2,4,V-01,False
1,Fries,<NA>,6,V-01,True


Orders with no line items: [5]
Known units: 2 | lines with unknown qty: 1


### Q6 — Validate

In [ ]:
assert len(lines) == 4
assert lines['qty'].sum() == 7
assert len(joined) == len(lines), 'the vendor join changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** Why is "one row per line item" the useful shape here? What question becomes easy that would be hard with one row per order?

**b)** In Q5, what is the practical difference between a missing quantity and a quantity of zero, and what breaks downstream if you conflate them?

a) One row per line item is the useful shape here as quantity belongs to an item on an order, not the order itself. As a result, the line item is the level where QTY has exactly one value per row. With one row per line item, the question of "how many units" becomes a lot easier, which would've been harder with one row per order. As now, a simple groupby and sum can tell us the answer. With one row per order, the lines would sit in a nested list, that would need to be exploded before it could be summed or join anything.

b) The practical difference between a missing quantity and a quantity of zero here is that zero means none were sold whereas missing means that we were not told. The Fries line on order 6 exists, however we are not aware of how many were sold. In this case, conflating it with 0 makes the total look complete and drags the average down, leaving no way to find which records may need fixing. Keeping the column nullable leaves the value as 'NA' and the qty_missing flag keeps the gap visible and countable.